# Notebook 01 - Locked publication training: Agent A (EfficientNet-B4)

This notebook trains Agent A only from the locked `train.csv` and `model_val.csv` publication artifacts. Checkpoint selection and early stopping use `model_val` exclusively. Full runs write the best model and reproducibility metadata under `artifacts/rescue/checkpoints`; `ARGUS_SMOKE_TEST=1` redirects every output to a temporary directory.


## 1. Install extras, imports, and shared helpers
Installs the training dependencies, locates the Argus modules, configures locked split/checkpoint roots, and defines hash, fingerprint, overlap, smoke-test, and run-metadata helpers.

In [ ]:
import sys, os
from pathlib import Path


def _find_ml_training():
    for root in (".", "..", "/kaggle/working", "/kaggle/input"):
        if not os.path.isdir(root):
            continue
        for dirpath, _dirs, files in os.walk(root):
            if "config.py" in files and "transforms.py" in files:
                if dirpath not in sys.path:
                    sys.path.insert(0, dirpath)
                return Path(dirpath).resolve()
    raise FileNotFoundError("Could not locate the Argus ml_training directory.")


ML_TRAINING_DIR = _find_ml_training()
REPO_ROOT = ML_TRAINING_DIR.parent
print("Argus ml_training discovered at:", ML_TRAINING_DIR)

from transforms import get_train_transform, get_eval_transform
from weighting import effective_number_weights
from config import EFFECTIVE_NUMBER_BETA

import sys, subprocess
print("NOTE: Kaggle Internet must be ON for package installation and full-run pretrained weights.")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "timm", "grad-cam", "torchmetrics"], check=True)

import glob, random, math, warnings, json, hashlib, tempfile, platform
import importlib.metadata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision
import timm
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, roc_auc_score

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SMOKE_TEST = os.environ.get("ARGUS_SMOKE_TEST", "0") == "1"
os.environ.setdefault("ARGUS_RUN_VISUALIZATIONS", "0")
RUN_VISUALIZATIONS = os.environ["ARGUS_RUN_VISUALIZATIONS"] == "1"
DEFAULT_ARTIFACT_ROOT = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else REPO_ROOT
ARTIFACT_ROOT = Path(os.environ.get("ARGUS_ARTIFACT_ROOT", str(DEFAULT_ARTIFACT_ROOT))).resolve()
SPLIT_DIR = Path(os.environ.get("ARGUS_SPLIT_DIR", str(REPO_ROOT / "artifacts" / "rescue" / "splits"))).resolve()
PUBLICATION_CHECKPOINT_DIR = ARTIFACT_ROOT / "artifacts" / "rescue" / "checkpoints"

if SMOKE_TEST:
    CHECKPOINT_DIR = Path(tempfile.mkdtemp(prefix="argus_publication_smoke_"))
    print("SMOKE TEST: checkpoints and metadata redirected to", CHECKPOINT_DIR)
else:
    CHECKPOINT_DIR = PUBLICATION_CHECKPOINT_DIR
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
RESUME_DIR = CHECKPOINT_DIR / "resume"
RESUME_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR = CHECKPOINT_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

ISIC_CLASSES = ["MEL", "NV", "BCC", "AK", "BKL", "DF", "VASC", "SCC"]
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
IMAGE_SIZE = 224
NUM_CLASSES = 8


def _sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _normalized_image_id(value):
    return Path(str(value).strip()).stem.lower()


def _class_counts(frame):
    return {str(k): int(v) for k, v in frame["label"].value_counts().sort_index().items()}


def _assert_no_forbidden_split_path_variables(namespace):
    blocked = ("risk" + "_dev", "final" + "_test")
    offenders = []

    def visit(name, value):
        if isinstance(value, Path):
            value = str(value)
        if isinstance(value, str):
            low = value.lower()
            if any(token in low for token in blocked) and (".csv" in low or "\\" in low or "/" in low):
                offenders.append(name)
        elif isinstance(value, dict):
            for key, item in value.items():
                visit(f"{name}[{key!r}]", item)
        elif isinstance(value, (list, tuple, set)):
            for index, item in enumerate(value):
                visit(f"{name}[{index}]", item)

    for name, value in namespace.items():
        if not name.startswith("__"):
            visit(name, value)
    assert not offenders, "Forbidden held-out split path found in notebook variables: " + ", ".join(offenders)


def _load_locked_training_splits(split_dir):
    train_path = split_dir / "train.csv"
    model_val_path = split_dir / "model_val.csv"
    summary_path = split_dir / "split_summary.json"
    fingerprint_path = split_dir / "dataset_fingerprint.json"
    for path in (train_path, model_val_path, summary_path, fingerprint_path):
        if not path.is_file():
            raise FileNotFoundError(f"Required locked publication artifact is missing: {path}")

    allowed_csv_names = {train_path.name, model_val_path.name}

    def keep_allowed_manifest_pairs(pairs):
        return {
            key: value for key, value in pairs
            if not (isinstance(key, str) and key.lower().endswith(".csv") and key not in allowed_csv_names)
        }

    with summary_path.open("r", encoding="utf-8") as handle:
        raw_summary = json.load(handle, object_pairs_hook=keep_allowed_manifest_pairs)
    with fingerprint_path.open("r", encoding="utf-8") as handle:
        fingerprint = json.load(handle)

    allowed = {"train": train_path, "model_val": model_val_path}
    allowed_hashes = {}
    allowed_summary = {}
    for split_name, csv_path in allowed.items():
        expected = raw_summary.get("csv_sha256", {}).get(csv_path.name)
        actual = _sha256_file(csv_path)
        assert expected and actual == expected, (
            f"Locked split hash mismatch for {csv_path.name}: expected={expected}, actual={actual}"
        )
        allowed_hashes[csv_path.name] = actual
        allowed_summary[split_name] = raw_summary["splits"][split_name]

    frames = {name: pd.read_csv(path) for name, path in allowed.items()}
    required = {"image", "label", "lesion_id", "lesion_group", "research_split"}
    for split_name, frame in frames.items():
        missing = sorted(required - set(frame.columns))
        assert not missing, f"{split_name}.csv missing required columns: {missing}"
        values = set(frame["research_split"].dropna().astype(str))
        assert values == {split_name}, f"{split_name}.csv has invalid research_split values: {sorted(values)}"
        expected_info = allowed_summary[split_name]
        assert len(frame) == int(expected_info["rows"]), f"{split_name}.csv row count disagrees with manifest"
        assert _class_counts(frame) == {str(k): int(v) for k, v in expected_info["class_counts"].items()}, (
            f"{split_name}.csv class counts disagree with manifest"
        )

    train_images = set(frames["train"]["image"].map(_normalized_image_id))
    val_images = set(frames["model_val"]["image"].map(_normalized_image_id))
    assert not (train_images & val_images), "train/model_val image overlap detected"

    def present(values):
        return {str(v).strip() for v in values if pd.notna(v) and str(v).strip()}

    train_lesions = present(frames["train"]["lesion_id"])
    val_lesions = present(frames["model_val"]["lesion_id"])
    assert not (train_lesions & val_lesions), "train/model_val lesion_id overlap detected"
    train_groups = present(frames["train"]["lesion_group"])
    val_groups = present(frames["model_val"]["lesion_group"])
    assert not (train_groups & val_groups), "train/model_val lesion_group overlap detected"

    assert fingerprint.get("split_protocol") == "publication_rescue_v1", "Unexpected dataset fingerprint protocol"
    assert fingerprint.get("image_col") == "image" and fingerprint.get("label_col") == "label", (
        "Dataset fingerprint column contract does not match the training notebooks"
    )
    assert fingerprint.get("lesion_id_col") == "lesion_id", "Dataset fingerprint lesion column mismatch"
    assert int(fingerprint["n_rows"]) == int(raw_summary["total_rows"]), (
        "Dataset fingerprint row count disagrees with split manifest"
    )
    aggregate_counts = {}
    for info in raw_summary["splits"].values():
        for label, count in info["class_counts"].items():
            aggregate_counts[str(label)] = aggregate_counts.get(str(label), 0) + int(count)
    assert aggregate_counts == {str(k): int(v) for k, v in fingerprint["label_counts"].items()}, (
        "Dataset fingerprint label counts disagree with split manifest"
    )
    assert float(fingerprint["lesion_id_coverage_pct"]) >= 95.0, "Lesion metadata coverage is below 95%"
    image_digest = str(fingerprint.get("image_id_sha256", ""))
    assert len(image_digest) == 64 and all(c in "0123456789abcdef" for c in image_digest.lower()), (
        "Dataset fingerprint image-id digest is invalid"
    )

    safe_manifest = {
        "total_rows": int(raw_summary["total_rows"]),
        "splits": allowed_summary,
        "csv_sha256": allowed_hashes,
    }
    return frames["train"], frames["model_val"], safe_manifest, fingerprint, _sha256_file(fingerprint_path)


def _discover_image_dir(root="/kaggle/input"):
    override = os.environ.get("ARGUS_IMAGE_DIR")
    if override:
        path = Path(override).resolve()
        assert path.is_dir(), f"ARGUS_IMAGE_DIR does not exist: {path}"
        return path
    best_dir, best_count = None, 0
    for dirpath, _dirs, files in os.walk(root):
        count = sum(
            name.lower().startswith("isic_") and name.lower().endswith((".jpg", ".jpeg"))
            for name in files
        )
        if count > best_count:
            best_dir, best_count = Path(dirpath), count
    assert best_dir is not None, f"Could not find ISIC image files under {root}"
    print("Discovered image directory:", best_dir, f"({best_count:,} images)")
    return best_dir


def _attach_image_paths(frame, image_dir):
    image_index = {}
    for path in image_dir.iterdir():
        if path.is_file() and path.suffix.lower() in (".jpg", ".jpeg", ".png"):
            image_index[_normalized_image_id(path.name)] = str(path)
    result = frame.copy()
    result["path"] = result["image"].map(lambda value: image_index.get(_normalized_image_id(value)))
    missing = int(result["path"].isna().sum())
    assert missing == 0, f"{missing} locked split image(s) are missing from {image_dir}"
    return result


def _tiny_class_subset(frame, rows_per_class):
    subset = frame.groupby("label", group_keys=False, sort=True).head(rows_per_class).copy()
    return subset.sort_values(["label", "image"], kind="mergesort").reset_index(drop=True)


def _git_commit():
    try:
        return subprocess.check_output(
            ["git", "rev-parse", "HEAD"], cwd=str(REPO_ROOT), text=True, stderr=subprocess.DEVNULL
        ).strip()
    except Exception:
        if SMOKE_TEST:
            return "unavailable-in-smoke-test"
        raise RuntimeError("Publication training requires a Git checkout so the run commit can be recorded.")


def _package_versions():
    packages = ["torch", "torchvision", "timm", "numpy", "pandas", "scikit-learn", "Pillow", "torchmetrics"]
    versions = {"python": platform.python_version()}
    for package in packages:
        try:
            versions[package] = importlib.metadata.version(package)
        except importlib.metadata.PackageNotFoundError:
            versions[package] = "not-installed"
    return versions


BEST_SELECTION = {}


def _selection_key(path):
    return str(Path(path).resolve())


def _record_selection(path, epoch, stage, metrics):
    record = {"epoch": int(epoch), "stage": stage, "validation_metrics": metrics}
    BEST_SELECTION[_selection_key(path)] = record
    sidecar = RESUME_DIR / (Path(path).stem + "_selection.json")
    sidecar.write_text(json.dumps(record, indent=2, sort_keys=True) + "\n", encoding="utf-8")


def _restore_selection(path):
    key = _selection_key(path)
    if key in BEST_SELECTION:
        return BEST_SELECTION[key]
    sidecar = RESUME_DIR / (Path(path).stem + "_selection.json")
    if sidecar.is_file():
        BEST_SELECTION[key] = json.loads(sidecar.read_text(encoding="utf-8"))
        return BEST_SELECTION[key]
    return None


def _write_run_metadata(agent_id, checkpoint_path, architecture, pretrained_source,
                        optimizer_config, scheduler_config, loss_config, sampling_config,
                        validation_metrics, transform_config):
    selection = _restore_selection(checkpoint_path)
    assert selection is not None and selection.get("epoch") is not None, (
        "Cannot publish run metadata without the checkpoint-selection epoch"
    )
    payload = {
        "agent": agent_id,
        "smoke_test": SMOKE_TEST,
        "git_commit": _git_commit(),
        "split_manifest_hashes": SPLIT_MANIFEST["csv_sha256"],
        "dataset_fingerprint": DATASET_FINGERPRINT,
        "dataset_fingerprint_file_sha256": DATASET_FINGERPRINT_SHA256,
        "model_architecture": architecture,
        "pretrained_checkpoint_source": pretrained_source,
        "pretrained_enabled_for_this_run": PRETRAINED_ENABLED,
        "random_seed": SEED,
        "optimizer": optimizer_config,
        "scheduler": scheduler_config,
        "loss": loss_config,
        "sampling_method": sampling_config,
        "epoch_selected": int(selection["epoch"]),
        "selection_stage": selection["stage"],
        "validation_metrics": validation_metrics,
        "checkpoint_sha256": _sha256_file(checkpoint_path),
        "checkpoint_path": str(Path(checkpoint_path)),
        "transform_configuration": transform_config,
        "package_versions": _package_versions(),
    }
    metadata_path = CHECKPOINT_DIR / f"{agent_id}_run_metadata.json"
    metadata_path.write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    print("Run metadata:", metadata_path)
    print("Checkpoint SHA-256:", payload["checkpoint_sha256"])
    return metadata_path


MODEL_NAME = "efficientnet_b4"
PRETRAINED_SOURCE = "timm efficientnet_b4 ImageNet pretrained weights"
PRETRAINED_ENABLED = not SMOKE_TEST
CKPT_PATH = str(CHECKPOINT_DIR / "agent_a_best.pth")
OUTPUT_DIR = str(FIGURE_DIR)


In [ ]:
# ===== LOCKED PUBLICATION RUN CONFIG =====
import config as _cfg
print("=" * 60)
print("ARGUS LOCKED PUBLICATION RUN")
print(f"  smoke_test={SMOKE_TEST} | device={DEVICE} | seed={SEED}")
print(f"  IMAGE_SIZE={_cfg.IMAGE_SIZE} | TRAINING_MODE={_cfg.TRAINING_MODE!r} | BETA={_cfg.EFFECTIVE_NUMBER_BETA}")
print(f"  STAGE_A/B epochs={_cfg.STAGE_A_EPOCHS}/{_cfg.STAGE_B_EPOCHS} | joint max={_cfg.PHASE2_MAX_EPOCHS} | patience={_cfg.PHASE2_PATIENCE}")
print(f"  checkpoint={CKPT_PATH}")
print(f"  resume_dir={RESUME_DIR}")
print("=" * 60)


## 2. Locked train/model-validation data and canonical transforms

Loads and verifies only the two development artifacts authorized for model fitting. Hashes, research roles, fingerprint aggregates, and image/lesion disjointness are checked before any image is opened.


In [ ]:
TRAIN_CSV = SPLIT_DIR / "train.csv"
MODEL_VAL_CSV = SPLIT_DIR / "model_val.csv"
train_df, model_val_df, SPLIT_MANIFEST, DATASET_FINGERPRINT, DATASET_FINGERPRINT_SHA256 = (
    _load_locked_training_splits(SPLIT_DIR)
)
_assert_no_forbidden_split_path_variables(globals())

IMAGE_DIR = _discover_image_dir()
train_df = _attach_image_paths(train_df, IMAGE_DIR)
model_val_df = _attach_image_paths(model_val_df, IMAGE_DIR)

if SMOKE_TEST:
    train_df = _tiny_class_subset(train_df, 1)
    model_val_df = _tiny_class_subset(model_val_df, 2)
    print(f"SMOKE TEST subset: train={len(train_df)} model_val={len(model_val_df)}")

for frame in (train_df, model_val_df):
    frame["_label"] = frame["label"].astype(np.int64)
    frame["_image"] = frame["image"].astype(str)

labels = pd.concat([train_df["_label"], model_val_df["_label"]], ignore_index=True).to_numpy()
df = pd.concat([train_df, model_val_df], ignore_index=True)
print("Locked train:", len(train_df), "| locked model_val:", len(model_val_df))
print("Allowed CSV hashes:", SPLIT_MANIFEST["csv_sha256"])
print("Dataset fingerprint SHA-256:", DATASET_FINGERPRINT_SHA256)


class ISICDataset(Dataset):
    """Reads locked publication rows. __getitem__ -> (tensor, label_int, image_path)."""
    def __init__(self, frame, transform):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        path = row["path"]
        image = self.transform(Image.open(path).convert("RGB"))
        return image, int(row["_label"]), path


train_tf = get_train_transform()
val_tf = get_eval_transform()
train_ds = ISICDataset(train_df, train_tf)
val_ds = ISICDataset(model_val_df, val_tf)

train_labels = train_df["_label"].values
class_counts = np.bincount(train_labels, minlength=NUM_CLASSES).astype(np.float64)
class_counts = np.clip(class_counts, 1, None)
class_weights = effective_number_weights(class_counts, beta=EFFECTIVE_NUMBER_BETA)
class_weights_t = torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)
print("Class weights:", dict(zip(ISIC_CLASSES, np.round(class_weights, 3))))

sample_weights = class_weights[train_labels]
sampler = WeightedRandomSampler(
    weights=torch.as_tensor(sample_weights, dtype=torch.double),
    num_samples=len(sample_weights), replacement=True,
)

BATCH_SIZE = min(8, len(train_ds)) if SMOKE_TEST else 32
train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, sampler=sampler,
    num_workers=0 if SMOKE_TEST else 2, pin_memory=(DEVICE == "cuda"),
    drop_last=not SMOKE_TEST,
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=0 if SMOKE_TEST else 2, pin_memory=(DEVICE == "cuda"),
)
print("Train batches:", len(train_loader), "| model_val batches:", len(val_loader))


## 3. Exploratory data analysis
Class-distribution bar chart and a 3x3 grid of sample images (one per class).

In [ ]:
counts_full = np.bincount(labels, minlength=NUM_CLASSES)

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(ISIC_CLASSES, counts_full, color=sns.color_palette('viridis', NUM_CLASSES))
ax.set_title('ISIC-2019 class distribution')
ax.set_ylabel('count')
for b, c in zip(bars, counts_full):
    ax.text(b.get_x() + b.get_width() / 2, c, str(int(c)), ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'class_distribution.png'), dpi=120)
plt.show()

# 3x3 sample grid: one example per class (8 classes), last cell a random extra
fig, axes = plt.subplots(3, 3, figsize=(9, 9))
axes = axes.ravel()
for i in range(9):
    if i < NUM_CLASSES:
        sub = df[df['_label'] == i]
        row = sub.iloc[0]
        title = ISIC_CLASSES[i]
    else:
        row = df.sample(1, random_state=7).iloc[0]
        title = 'random: ' + ISIC_CLASSES[int(row['_label'])]
    p = os.path.join(IMAGE_DIR, row['_image'] + '.jpg')
    axes[i].imshow(Image.open(p).convert('RGB'))
    axes[i].set_title(title, fontsize=10)
    axes[i].axis('off')
plt.suptitle('Sample lesion images', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'sample_grid.png'), dpi=120, bbox_inches='tight')
plt.show()

## 4. Focal loss
Inline `FocalLoss` with the publication-correct formula: `p_t` is always the true-class softmax probability, and optional `alpha[target]` scales only the base cross-entropy term.

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=None):
        super().__init__()
        self.gamma = gamma
        if alpha is not None and not torch.is_tensor(alpha):
            alpha = torch.tensor(alpha, dtype=torch.float32)
        self.register_buffer('alpha', alpha if alpha is not None else None)

    def forward(self, logits, targets):
        logp = F.log_softmax(logits, dim=1)
        log_pt = logp.gather(dim=1, index=targets.unsqueeze(1)).squeeze(1)
        pt = log_pt.exp()
        base_ce = -log_pt
        if self.alpha is not None:
            at = self.alpha.to(device=logits.device, dtype=logits.dtype)[targets]
            base_ce = base_ce * at
        focal = (1 - pt) ** self.gamma * base_ce
        return focal.mean()


criterion = FocalLoss(gamma=2.0, alpha=None)
print('FocalLoss ready (gamma=2.0, alpha=None; train_loader handles class balancing).')

## 5. Model + Phase 1 (train classifier head only)
Builds a pretrained EfficientNet-B4 via `timm`, freezes the backbone, and trains only the classifier head for ~5 epochs with `AdamW(lr=1e-3)` and mixed precision (`autocast` + `GradScaler`). Validation reports balanced accuracy and macro one-vs-rest AUC.

In [ ]:
from torchmetrics.classification import MulticlassAUROC

model = timm.create_model(MODEL_NAME, pretrained=PRETRAINED_ENABLED, num_classes=NUM_CLASSES)
model = model.to(DEVICE)

# timm exposes the head via get_classifier(); freeze the backbone, keep the head trainable.
classifier = model.get_classifier()
for p in model.parameters():
    p.requires_grad = False
for p in classifier.parameters():
    p.requires_grad = True
print('Phase 1 trainable params:', sum(p.numel() for p in model.parameters() if p.requires_grad))

scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == 'cuda'))
auroc_metric = MulticlassAUROC(num_classes=NUM_CLASSES, average='macro').to(DEVICE)


def run_epoch(loader, optimizer=None):
    """One pass over loader. Returns (loss, balanced_acc, macro_auc, probs, targets)."""
    train_mode = optimizer is not None
    model.train(train_mode)
    total_loss, n = 0.0, 0
    all_probs, all_targets = [], []
    for imgs, ys, _paths in loader:
        imgs = imgs.to(DEVICE, non_blocking=True)
        ys = ys.to(DEVICE, non_blocking=True)
        with torch.set_grad_enabled(train_mode):
            with torch.cuda.amp.autocast(enabled=(DEVICE == 'cuda')):
                logits = model(imgs)
                loss = criterion(logits, ys)
            if train_mode:
                optimizer.zero_grad()
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
        total_loss += loss.item() * imgs.size(0)
        n += imgs.size(0)
        all_probs.append(torch.softmax(logits.float(), dim=1).detach().cpu())
        all_targets.append(ys.detach().cpu())
    probs = torch.cat(all_probs).numpy()
    targets = torch.cat(all_targets).numpy()
    preds = probs.argmax(axis=1)
    bal_acc = balanced_accuracy_score(targets, preds)
    try:
        macro_auc = roc_auc_score(targets, probs, multi_class='ovr', average='macro',
                                  labels=list(range(NUM_CLASSES)))
    except ValueError:
        macro_auc = float('nan')
    return total_loss / max(n, 1), bal_acc, macro_auc, probs, targets


history = {'train_loss': [], 'val_loss': [], 'val_bal_acc': [], 'val_auc': []}

PHASE1_EPOCHS = 1 if SMOKE_TEST else 5
opt1 = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-3, weight_decay=1e-4)
for ep in range(1, PHASE1_EPOCHS + 1):
    tr_loss, _, _, _, _ = run_epoch(train_loader, opt1)
    val_loss, val_bal, val_auc, _, _ = run_epoch(val_loader, None)
    history['train_loss'].append(tr_loss)
    history['val_loss'].append(val_loss)
    history['val_bal_acc'].append(val_bal)
    history['val_auc'].append(val_auc)
    if SMOKE_TEST:
        torch.save(model.state_dict(), CKPT_PATH)
        _record_selection(CKPT_PATH, ep, 'smoke-head', {'balanced_accuracy': float(val_bal), 'macro_auc': float(val_auc), 'loss': float(val_loss)})
        print('SMOKE TEST: completed exactly one training batch and saved temporary checkpoint:', CKPT_PATH)
    print('[P1 %d/%d] train_loss=%.4f val_loss=%.4f val_bal_acc=%.4f val_macroAUC=%.4f'
          % (ep, PHASE1_EPOCHS, tr_loss, val_loss, val_bal, val_auc))


## 6. Phase 2 - Decoupled (Kang et al.) or Joint fine-tune (config-driven)

Branches on `config.TRAINING_MODE`:

- **`decoupled`** (Kang et al., 2020): **Stage A** learns representations on the **full** network with **instance-balanced** sampling (a plain shuffled `DataLoader` over `train_ds`, *no* `WeightedRandomSampler`) and an **unweighted** focal loss (`alpha=None`), `AdamW(lr=1e-5)` + `CosineAnnealingLR(T_max=STAGE_A_EPOCHS)`, early-stopping on `model_val` macro-AUC; the best Stage-A weights are saved under the new rescue resume directory and reloaded before Stage B. **Stage B** freezes the backbone (`freeze_all_but_classifier`) and retrains **only the classifier head** with the configured Stage-B imbalance policy (publication default: weighted sampler, `FocalLoss(alpha=None)`), `AdamW(lr=1e-3)` + `CosineAnnealingLR(T_max=STAGE_B_EPOCHS)`. After Stage B, `assert_frozen_unchanged` proves every backbone param is bit-identical.
- **`joint`**: the original single-stage path - partial unfreeze of the last `PHASE2_UNFREEZE_BLOCKS` blocks + head, publication config guarding against sampler+alpha double rebalancing, `AdamW(lr=1e-5)` + `CosineAnnealingLR`, early stopping up to `PHASE2_MAX_EPOCHS` (patience `PHASE2_PATIENCE`).

An independent `USE_LOGIT_ADJUSTMENT` toggle wraps the active training loss with `LogitAdjustedLoss` (`logits + tau*log(prior)`, empirical priors). The final best checkpoint is selected only by `model_val` macro-AUC and written to `artifacts/rescue/checkpoints/agent_a_best.pth`.

In [ ]:
# Phase 2 (config-driven): Kang et al. (2020) DECOUPLED two-stage training, OR the
# original single-stage JOINT fine-tune, plus an INDEPENDENT logit-adjustment toggle.
# AUDIT history (1.1 / 1.2c): early stopping on val macro-AUC over the PHASE2_* budget is
# preserved on every path, and the best-by-macro-AUC checkpoint still lands at CKPT_PATH.
from config import (TRAINING_MODE, STAGE_A_EPOCHS, STAGE_B_EPOCHS,
                    PHASE2_MAX_EPOCHS, PHASE2_PATIENCE,
                    USE_LOGIT_ADJUSTMENT, LOGIT_ADJUSTMENT_TAU, FOCAL_LOSS_GAMMA,
                    USE_WEIGHTED_SAMPLER_STAGE_B, USE_CLASS_WEIGHTED_LOSS_STAGE_B,
                    ALLOW_LEGACY_STAGE_B_DOUBLE_REBALANCING)
from training_utils import (class_priors_from_counts, LogitAdjustedLoss,
                            freeze_all_but_classifier, snapshot_frozen_params,
                            assert_frozen_unchanged, freeze_backbone_bn,
                            assert_single_stage_b_rebalancing,
                            save_resumable, load_resumable)

# EMPIRICAL (un-rebalanced) class priors from the per-class TRAIN counts this nb computed.
priors = class_priors_from_counts(class_counts)


def _run_stage(loader, optimizer, scheduler, max_epochs, patience, ckpt_path, tag):
    """One early-stopped training stage over the GLOBAL model / criterion.

    Wraps the notebook's OWN run_epoch(loader, optimizer) per-epoch train/eval
    (run_epoch reads the global `criterion`, `model` and `scaler`), early-stops on
    val macro-AUC, appends every epoch to the shared `history`, saves the
    best-by-macro-AUC checkpoint to ckpt_path, and returns the best macro-AUC.
    The SAME helper drives Stage A, Stage B AND the joint path, so there is exactly
    ONE training/early-stopping loop in this notebook.
    """
    best_auc = -1.0
    epochs_no_improve = 0
    last_epoch = 0
    start_epoch = 1
    resume_path = str(RESUME_DIR / ("agent_a_" + tag.lower() + "_resume.pth"))
    if SMOKE_TEST:
        print("[%s] skipped in smoke mode; head smoke batch already completed." % tag)
        return -1.0
    _r = load_resumable(resume_path, model, optimizer, scheduler, scaler, map_location=DEVICE)
    if _r is not None:
        start_epoch, best_auc, epochs_no_improve = _r["start_epoch"], _r["best_auc"], _r["epochs_no_improve"]
        print('[%s] RESUMED from %s -> start epoch %d (best=%.4f, no_improve=%d)'
              % (tag, resume_path, start_epoch, best_auc, epochs_no_improve))
    for ep in range(start_epoch, max_epochs + 1):
        last_epoch = ep
        tr_loss, _, _, _, _ = run_epoch(loader, optimizer)
        val_loss, val_bal, val_auc, _, _ = run_epoch(val_loader, None)
        if scheduler is not None:
            scheduler.step()
        history['train_loss'].append(tr_loss)
        history['val_loss'].append(val_loss)
        history['val_bal_acc'].append(val_bal)
        history['val_auc'].append(val_auc)
        improved = ''
        if not math.isnan(val_auc) and val_auc > best_auc:
            best_auc = val_auc
            epochs_no_improve = 0
            torch.save(model.state_dict(), ckpt_path)
            _record_selection(ckpt_path, ep, tag, {'loss': float(val_loss), 'balanced_accuracy': float(val_bal), 'macro_auc': float(val_auc)})
            improved = '  <-- saved best'
        else:
            epochs_no_improve += 1
        print('[%s %d/%d] train_loss=%.4f val_loss=%.4f val_bal_acc=%.4f val_macroAUC=%.4f lr=%.2e no_improve=%d%s'
              % (tag, ep, max_epochs, tr_loss, val_loss, val_bal, val_auc,
                 optimizer.param_groups[0]['lr'], epochs_no_improve, improved))
        save_resumable(resume_path, tag, ep, model, optimizer, scheduler, scaler, best_auc, epochs_no_improve)
        if epochs_no_improve >= patience:
            print('Early stopping %s at epoch %d (no val macro-AUC gain for %d epochs). Best=%.4f'
                  % (tag, ep, patience, best_auc))
            break
    print('[%s] best macro-AUC=%.4f after %d epochs | checkpoint: %s'
          % (tag, best_auc, last_epoch, ckpt_path))
    return best_auc


if TRAINING_MODE == 'decoupled':
    # ===================== Stage A: representation learning =====================
    #  * UNFREEZE THE FULL NETWORK (every param trains).
    for p in model.parameters():
        p.requires_grad_(True)
    stageA_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print('[StageA] full-network trainable params: %d (%.2fM)'
          % (stageA_trainable, stageA_trainable / 1e6))

    #  * INSTANCE-BALANCED sampling: a PLAIN shuffled DataLoader over train_ds
    #    (NO WeightedRandomSampler; we do NOT reuse the weighted train_loader here).
    stageA_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                               num_workers=2, pin_memory=True, drop_last=True)

    #  * UNWEIGHTED focal loss (no class weighting) for representation learning.
    criterion = FocalLoss(gamma=FOCAL_LOSS_GAMMA, alpha=None).to(DEVICE)

    #  * optimizer over ALL params, LOW lr, CosineAnnealingLR(T_max=STAGE_A_EPOCHS).
    optA = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                             lr=1e-5, weight_decay=1e-4)
    schedA = torch.optim.lr_scheduler.CosineAnnealingLR(optA, T_max=STAGE_A_EPOCHS)

    #  * early-stop up to STAGE_A_EPOCHS (patience PHASE2_PATIENCE); best weights -> separate file.
    STAGEA_CKPT = str(RESUME_DIR / 'agent_a_stage_a_best.pth')
    _run_stage(stageA_loader, optA, schedA, STAGE_A_EPOCHS, PHASE2_PATIENCE,
               STAGEA_CKPT, 'StageA')

    #  * reload the BEST Stage-A weights before Stage B (fallback keeps current weights).
    if not os.path.exists(STAGEA_CKPT):
        torch.save(model.state_dict(), STAGEA_CKPT)
        print('[StageA] no improving checkpoint; saved final-state weights as fallback.')
    model.load_state_dict(torch.load(STAGEA_CKPT, map_location=DEVICE))
    print('[StageA] reloaded best Stage-A weights from', STAGEA_CKPT)

    # ===================== Stage B: classifier re-balancing =====================
    #  * freeze everything but the classifier head.
    n_head, n_total = freeze_all_but_classifier(model)
    n_bn_frozen = freeze_backbone_bn(model)  # cRT: also freeze backbone BN running stats (no-op for ViT/LayerNorm)
    print(f"[Stage B] froze {n_bn_frozen} BatchNorm module(s) so running stats are held too.")
    print('[StageB] head-only trainable params: %d (of %d total)' % (n_head, n_total))
    snap = snapshot_frozen_params(model)  # capture BEFORE Stage B training.

    #  * Publication default: weighted sampler OR weighted focal loss, never both.
    assert_single_stage_b_rebalancing(USE_WEIGHTED_SAMPLER_STAGE_B,
                                      USE_CLASS_WEIGHTED_LOSS_STAGE_B,
                                      ALLOW_LEGACY_STAGE_B_DOUBLE_REBALANCING)
    stageB_loader = train_loader if USE_WEIGHTED_SAMPLER_STAGE_B else stageA_loader
    stageB_alpha = class_weights_t if USE_CLASS_WEIGHTED_LOSS_STAGE_B else None
    base_lossB = FocalLoss(gamma=FOCAL_LOSS_GAMMA, alpha=stageB_alpha)
    if USE_LOGIT_ADJUSTMENT:
        criterion = LogitAdjustedLoss(base_lossB, priors, LOGIT_ADJUSTMENT_TAU).to(DEVICE)
        print('[LOGIT ADJ] Stage B loss wrapped with tau*log(prior), tau=%.2f.' % LOGIT_ADJUSTMENT_TAU)
    else:
        criterion = base_lossB.to(DEVICE)

    #  * optimizer over ONLY the trainable (head) params, higher lr, CosineAnnealingLR(T_max=STAGE_B_EPOCHS).
    optB = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                             lr=1e-3, weight_decay=1e-4)
    schedB = torch.optim.lr_scheduler.CosineAnnealingLR(optB, T_max=STAGE_B_EPOCHS)

    #  * early-stop up to STAGE_B_EPOCHS, saving the FINAL best checkpoint to CKPT_PATH.
    best_auc = _run_stage(stageB_loader, optB, schedB, STAGE_B_EPOCHS, PHASE2_PATIENCE,
                          CKPT_PATH, 'StageB')

    #  * PROVE the freeze held: every backbone param bit-identical after Stage B.
    n_verified = assert_frozen_unchanged(model, snap)
    print('[FREEZE VERIFIED] %d backbone params bit-identical after Stage B.' % n_verified)

else:  # 'joint' — the ORIGINAL single-stage Phase-2 fine-tune; behavior unchanged.
    # Unfreeze the last N backbone blocks + classifier head, then fine-tune.
    # AUDIT (1.1): the pre-audit version ran a FIXED 15 epochs with no early stopping;
    # AUDIT (1.2c) fix extended the budget and added early stopping on val macro-AUC.
    # Publication default: unweighted focal loss; Stage-B rebalancing is controlled by config.
    # The unfreeze depth is intentionally left unchanged here.
    PHASE2_UNFREEZE_BLOCKS = 2   # EfficientNet-B4 has 7 stages (blocks[0..6]); the last 2 are its heaviest.
    for p in model.parameters():
        p.requires_grad = False
    if hasattr(model, 'blocks'):
        for blk in model.blocks[-PHASE2_UNFREEZE_BLOCKS:]:
            for p in blk.parameters():
                p.requires_grad = True
    for p in model.get_classifier().parameters():
        p.requires_grad = True
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print('Phase 2 trainable params: %d (%.2fM)' % (trainable, trainable / 1e6))
    print('  AUDIT 1.2d: if this is < 5M, raise PHASE2_UNFREEZE_BLOCKS to add capacity.')

    # The global `criterion` (cell 4: weighted FocalLoss) is the joint loss; wrap it only
    # if logit adjustment is toggled on, otherwise leave it exactly as-is.
    if USE_LOGIT_ADJUSTMENT:
        criterion = LogitAdjustedLoss(criterion, priors, LOGIT_ADJUSTMENT_TAU).to(DEVICE)
        print('[LOGIT ADJ] joint loss wrapped with tau*log(prior), tau=%.2f.' % LOGIT_ADJUSTMENT_TAU)

    opt2 = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-5, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt2, T_max=PHASE2_MAX_EPOCHS)
    best_auc = _run_stage(train_loader, opt2, sched, PHASE2_MAX_EPOCHS, PHASE2_PATIENCE,
                          CKPT_PATH, 'P2')

# Fallback: ensure a checkpoint exists even if val macro-AUC was NaN throughout.
if not os.path.exists(CKPT_PATH):
    torch.save(model.state_dict(), CKPT_PATH)
    print('Saved final-state checkpoint as fallback.')
print('Phase 2 complete (mode=%s). Best macro-AUC=%.4f | checkpoint: %s'
      % (TRAINING_MODE, best_auc, CKPT_PATH))


## 7. Training curves
Loss, balanced accuracy, and macro-AUC across both phases (dashed line marks the Phase 1 -> Phase 2 boundary).

In [ ]:
epochs = range(1, len(history['train_loss']) + 1)
split = PHASE1_EPOCHS + 0.5

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(list(epochs), history['train_loss'], marker='o', label='train')
axes[0].plot(list(epochs), history['val_loss'], marker='o', label='val')
axes[0].axvline(split, color='gray', ls='--', alpha=0.6)
axes[0].set_title('Loss'); axes[0].set_xlabel('epoch'); axes[0].legend()

axes[1].plot(list(epochs), history['val_bal_acc'], marker='o', color='tab:green')
axes[1].axvline(split, color='gray', ls='--', alpha=0.6)
axes[1].set_title('Val balanced accuracy'); axes[1].set_xlabel('epoch')

axes[2].plot(list(epochs), history['val_auc'], marker='o', color='tab:purple')
axes[2].axvline(split, color='gray', ls='--', alpha=0.6)
axes[2].set_title('Val macro AUC'); axes[2].set_xlabel('epoch')

plt.suptitle('Agent A training (dashed line = phase 1 -> phase 2)')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'training_curves.png'), dpi=120)
plt.show()

## 8. Confusion matrix + per-class metrics
Loads the best checkpoint, runs the validation set, and reports a row-normalized confusion-matrix heatmap and a per-class metric table.

In [ ]:
from sklearn.metrics import classification_report

model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
_, val_bal, val_auc, val_probs, val_targets = run_epoch(val_loader, None)
val_preds = val_probs.argmax(axis=1)
print('Best-checkpoint model_val balanced acc=%.4f  macro AUC=%.4f' % (val_bal, val_auc))

cm = confusion_matrix(val_targets, val_preds, labels=list(range(NUM_CLASSES)))
cm_norm = cm.astype(float) / np.clip(cm.sum(axis=1, keepdims=True), 1, None)

fig, ax = plt.subplots(figsize=(8, 6.5))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=ISIC_CLASSES, yticklabels=ISIC_CLASSES, ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('Agent A confusion matrix (row-normalized)')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'confusion_matrix.png'), dpi=120)
plt.show()

report = classification_report(val_targets, val_preds, labels=list(range(NUM_CLASSES)),
                               target_names=ISIC_CLASSES, output_dict=True, zero_division=0)
report_df = pd.DataFrame(report).transpose().round(4)
print(report_df)

_write_run_metadata(
    agent_id="agent_a",
    checkpoint_path=CKPT_PATH,
    architecture=MODEL_NAME,
    pretrained_source=PRETRAINED_SOURCE,
    optimizer_config={"name": "AdamW", "head_lr": 1e-3, "representation_lr": 1e-5, "weight_decay": 1e-4},
    scheduler_config={"name": "CosineAnnealingLR", "stage_a_t_max": STAGE_A_EPOCHS, "stage_b_t_max": STAGE_B_EPOCHS},
    loss_config={"name": "FocalLoss", "gamma": float(FOCAL_LOSS_GAMMA), "stage_a_alpha": None, "stage_b_class_weighted": bool(USE_CLASS_WEIGHTED_LOSS_STAGE_B)},
    sampling_config={"stage_a": "ordinary shuffled DataLoader", "stage_b": "WeightedRandomSampler" if USE_WEIGHTED_SAMPLER_STAGE_B else "ordinary shuffled DataLoader"},
    validation_metrics={"balanced_accuracy": float(val_bal), "macro_auc": float(val_auc)},
    transform_config={"train": repr(train_tf), "model_val": repr(val_tf), "image_size": IMAGE_SIZE, "mean": IMAGENET_MEAN, "std": IMAGENET_STD},
)


## 8b. Per-class validation report (audit item 1.3)
Explicit balanced accuracy and per-class recall on the held-out split, with a flag for any class whose recall is below 0.40, so minority-class behaviour is visible at a glance after a Kaggle run. Reuses `val_targets`/`val_preds` from the best checkpoint above.

In [ ]:
from sklearn.metrics import classification_report, balanced_accuracy_score, recall_score

CLASS_NAMES = ISIC_CLASSES
bal_acc = balanced_accuracy_score(val_targets, val_preds)
print('Final validation balanced accuracy: %.4f' % bal_acc)
print('\nPer-class report:')
print(classification_report(val_targets, val_preds, labels=list(range(NUM_CLASSES)),
                            target_names=CLASS_NAMES, digits=4, zero_division=0))
print('\nFlag: any class recall below 0.40?')
recalls = recall_score(val_targets, val_preds, average=None, labels=list(range(NUM_CLASSES)), zero_division=0)
for cls, rec in zip(CLASS_NAMES, recalls):
    flag = '  <-- LOW' if rec < 0.40 else ''
    print('  %-4s recall=%.4f%s' % (cls, rec, flag))


## 9. Grad-CAM++ explanations
Optional and disabled by default during publication training. Set `ARGUS_RUN_VISUALIZATIONS=1` to overlay Grad-CAM++ saliency on 9 validation images. The target layer is resolved robustly (`model.conv_head`, else `model.blocks[-1]`).

In [ ]:
if SMOKE_TEST or not RUN_VISUALIZATIONS:
    print("Optional Grad-CAM generation skipped (set ARGUS_RUN_VISUALIZATIONS=1 to enable).")
else:
    from pytorch_grad_cam import GradCAMPlusPlus
    from pytorch_grad_cam.utils.image import show_cam_on_image
    from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

    # Resolve a sensible conv target layer for EfficientNet
    if hasattr(model, 'conv_head') and model.conv_head is not None:
        target_layers = [model.conv_head]
    elif hasattr(model, 'blocks'):
        target_layers = [model.blocks[-1]]
    else:
        target_layers = [list(model.modules())[-3]]
    print('Grad-CAM target layer:', target_layers[0].__class__.__name__)

    mean = np.array(IMAGENET_MEAN, dtype=np.float32)
    std = np.array(IMAGENET_STD, dtype=np.float32)

    vis_samples = model_val_df.sample(9, random_state=123).reset_index(drop=True)
    model.eval()
    cam = GradCAMPlusPlus(model=model, target_layers=target_layers)

    fig, axes = plt.subplots(3, 3, figsize=(11, 11))
    axes = axes.ravel()
    for i in range(9):
        row = vis_samples.iloc[i]
        p = os.path.join(IMAGE_DIR, row['_image'] + '.jpg')
        pil = Image.open(p).convert('RGB')
        inp = val_tf(pil).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            pred = int(model(inp).argmax(dim=1).item())
        grayscale = cam(input_tensor=inp, targets=[ClassifierOutputTarget(pred)])[0]
        # de-normalize the input image for display
        rgb = inp[0].detach().cpu().numpy().transpose(1, 2, 0)
        rgb = np.clip(rgb * std + mean, 0, 1).astype(np.float32)
        overlay = show_cam_on_image(rgb, grayscale, use_rgb=True)
        true_idx = int(row['_label'])
        axes[i].imshow(overlay)
        axes[i].set_title('true=%s  pred=%s' % (ISIC_CLASSES[true_idx], ISIC_CLASSES[pred]),
                          fontsize=9, color=('green' if pred == true_idx else 'red'))
        axes[i].axis('off')
    plt.suptitle('Grad-CAM++ on validation images', y=1.01)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'gradcam_grid.png'), dpi=120, bbox_inches='tight')
    plt.show()


## Publication outputs

The full run writes `artifacts/rescue/checkpoints/agent_a_best.pth`, resumable stage state under `artifacts/rescue/checkpoints/resume/`, and `artifacts/rescue/checkpoints/agent_a_run_metadata.json`.
